# Developing data simulation

The objective of this notebook is helping to develop step by step the simulation from `R` into `python`

The first objective will be the creation of a simulation possibility based only on continuous features. No categorical will be considered.

In [1]:
import torch

In [2]:
means = torch.tensor([3.5,-3.5], dtype=torch.float64)
covs = torch.tensor([[1,-0.5],[-0.5,1]], dtype=torch.float64)
mvn = torch.distributions.MultivariateNormal(means, covariance_matrix=covs)

In a 2 dimensional multivariate distribution, we have a vector $X$ of two RV
$$
X = \begin{pmatrix} X_1 \\ X_2 \end{pmatrix}
$$
Mean and VCOV matrix are given by
$$
\mu = \begin{pmatrix} 3.5 \\ -3.5 \end{pmatrix}, \; 
\Sigma = \begin{pmatrix}
1 & -\frac{1}{2} \\
-\frac{1}{2} & 1
\end{pmatrix}
$$

This means when sampling we will only get positive numbers for $X_1$ realizations and negative numbers for $X_2$ realizations

In [3]:
sample_2d = mvn.sample((3,))
print(sample_2d.shape)
sample_2d

torch.Size([3, 2])


tensor([[ 4.0739, -2.5579],
        [ 3.5667, -2.2065],
        [ 4.5279, -4.0313]], dtype=torch.float64)

So in a sample size of `(n,)`, the shape will be `[n, k]` with `k` the number of random variables composing $X$.

With $x^{(r)}_j$ the $j$-th sampled realization of $X_r$, the output looks like
$$
\texttt{sample} = \begin{pmatrix}
    x^{(1)}_1 & \cdots & x^{(k)}_1 \\
    \vdots & \ddots & \vdots \\
    x^{(1)}_n & \cdots & x^{(k)}_n
\end{pmatrix} \in \mathbb{R}^{n \times k}
$$

## Notes on the R-Algorithm

### Calls:

First call is made for the init_population, by

```r
res <- genCreditData(
  #################################### DIMENSIONALITY
  n                = init_sample,  # - sample size = 100
  bad_ratio        = bad_ratio,    # - BAD ratio (if all D = 0) = 0.7
  k_con            = num_feats,    # - no. continuous features = 2
  k_cat            = 0,            # - no. categorical features
  k_bin            = 0,            # - no. binary features
  k_noise          = num_noise,    # - no. white-noise features = 0
  #################################### CONTINUOUS FEATURES
  con_nonlinear    = 0.0,       # - share of nonlinear transformations
  con_mean_bad_dif = mean_dif,  # - mean difference between classes = c(2, 1)
  con_var_bad_dif  = var_dif,   # - share of var/covar difference between classes = 0.5
  con_noise_var    = noise_var, # - variance of noise = 0
  covars           = covars,    # - variance-covariance matrices = list(matrix(c(1,  0.2,  0.2, 1), nrow = 2), matrix(c(1, -0.2, -0.2, 1), nrow = 2))
  #################################### MIXTURE OF GAUSSIANS
  mixture      = mixture,       # - mixture of two Gaussians = FALSE
  mix_mean_dif = mix_mean_dif,  # - mean difference between components  = 5
  mix_var_dif  = mix_var_dif,   # - share of var/covar difference between components = 0
  #################################### OTHER PARAMETERS
  seed             = seed,   # - random seed
  verbose          = F,      # - displaying feedback
  encode_factors   = F)      # - encoding of categorical features
```

`matrix(c(1,  0.2,  0.2, 1)` liefert
$$
\begin{pmatrix}
    1 & 0.2 \\
    0.2 & 1
\end{pmatrix}
$$

The next call is done this way:
```r
new_applicants <- genCreditData(n = sample_size, replicate = res, seed = seed + g)$data
```

Which is a quick way to replicate the arguments of the last call, based on this object (all names are set as objects - which are "`names`" in R)
```r
list(n                = n,
     k_cat            = k_cat,
     k_bin            = k_bin,
     k_noise          = k_noise,
     bad_ratio        = bad_ratio,
     con_mean_bad_dif = con_mean_bad_dif,
     con_var_bad_dif  = con_var_bad_dif,
     con_nonlinear    = con_nonlinear,
     con_noise_var    = con_noise_var,
     mixture          = mixture,
     mix_mean_dif     = mix_mean_dif,
     mix_var_dif      = mix_var_dif,
     cat_levels       = cat_levels,
     cat_var_share    = cat_var_share,
     cat_nonlinear    = cat_nonlinear,
     cat_noise_var    = cat_noise_var,
     bin_prob         = bin_prob,
     bin_mean_bad_dif = bin_mean_bad_dif,
     bin_bad_ratio    = bin_bad_ratio,
     bin_mean_con_dif = bin_mean_con_dif,
     bin_var_bad_dif  = bin_var_bad_dif,
     bin_noise_var    = bin_noise_var,
     encode_factors   = encode_factors,
     verbose          = verbose,
     seed             = seed)
```

however n and seed are not "taken over" - the rest they are

### Relevant parts of the algorithm

1. Set `combo_bad_ratio <- 0.7` and `combo_count <- n`
2. Compute "good" and "bad" sample sizes, as $n \cdot 0.7$ und $n \cdot (1-0.7)$ correspondingly. Ties are solved by 50/50 chance of doing good = n - bad or bad = n - good.
3. Set `mu_1 = c(0,0)` and therefore `mu_2 = c(1,2)=con_mean_bad_dif` (see line 249)

In [239]:
from typing import List, Optional, Tuple, Union
import torch
def random_vcov_matrix(
        k: int,
        generator: Optional[torch.Generator] = None,
        var_range: Tuple[float, float] = (0.0, 1.0),
        prefer_normal_base_sampling: bool = True,
        device: torch.device = torch.get_default_device(),
        dtype: torch.dtype = torch.get_default_dtype(),
        eps: float = 1e-6
    ) -> torch.Tensor:
    """Generate a random positive definite covariance matrix.

    The covariance matrix is produced by:

    1. Generating a base sampling of a matrix :math:`A` (normal or uniform).
    2. Creating a correlation matrix via cosine similarity between the row vectors
       of the base sampling:

       .. math::

          C = (c_{i,j}) = \\left( \\frac{\\langle A_i, A_j \\rangle}{\\|A_i\\| \\|A_j\\|} \\right)

    3. Sampling a variance vector uniformly in ``var_range``, which is used to
       rescale the correlation matrix.
    4. Ensuring positive definiteness via addition of a small diagonal
       perturbation ``eps``.
    5. Make sure exact symmetry, so that rounding point instability does not
       lead to unsymmetric results. 

    Args:
        k (int): Dimension of the covariance matrix.
        generator (torch.Generator, optional): Random number generator for reproducibility.
        var_range (Tuple[float, float], optional): Range for diagonal variances. Defaults to (0.0, 1.0).
        prefer_normal_base_sampling (bool, optional): If True, use normal distribution for base sampling.
            If False, use uniform distribution. Defaults to True.
        device (torch.device, optional): Device on which to allocate the tensor.
            Defaults to ``torch.get_default_device()``.
        dtype (torch.dtype, optional): Data type of the returned tensor.
            Defaults to ``torch.get_default_dtype()``.
        eps (float, optional): Small positive value added to the diagonal to ensure positive definiteness.
            Defaults to 1e-6.

    Returns:
        torch.Tensor: A symmetric, positive definite covariance matrix of shape ``(k, k)``.

    Raises:
        ValueError: If ``var_range`` is not a valid (min, max) tuple.

    Example:
        >>> g = torch.Generator().manual_seed(42)
        >>> cov = random_vcov_matrix(4, generator=g)
        >>> cov.shape
        torch.Size([4, 4])
    """

    # Step 1: Generate base sampling
    if prefer_normal_base_sampling:
        A = torch.randn((k, k), generator=generator, dtype=dtype, device=device) # random normal matrix, sparser correlations for high k
    else:
        A = 2*torch.rand((k,k), generator=generator, dtype = dtype, device=device) - 1 # random uniform matrix, correlations closer to 0, the higher k

    # Step 2: Define correlation matrix from base sampling
    Q = A @ A.T # Make sure of symmetry while using full randomness
    D = torch.sqrt(torch.diag(Q)) # Help vector for normalization
    corr_mat = Q / torch.outer(D, D) # corr_mat[i, j] = cosine_similarity(A[i], A[j]), so range [-1, 1] guaranteed

    # Step 3: Rescale corr_mat with sampled variances
    ## Variance sampling from uniform distribution
    variances = torch.rand(k, generator=generator, dtype = dtype, device=device) * (var_range[1] - var_range[0]) + var_range[0]
    ## Rescaling via outer prouct of standard deviations
    stds = torch.sqrt(variances)
    norm_factors_pearson_corr = torch.outer(stds, stds) # guaranteed to be symmetric, denominators of pearson correlation
    vcov = corr_mat * norm_factors_pearson_corr

    # Step 4: Avoid semi positive definitness of the matrix
    vcov = vcov + eps * torch.eye(k, device=device, dtype=dtype)

    # Step 5: Ensure **exact** symmetry without compromising randomness
    i, j = torch.tril_indices(k, k, offset=-1)
    vcov[i, j] = vcov[j, i]

    
    return vcov

def eigen_decomp_proj_to_pd(
    mat: torch.Tensor,
    eps: float = 1e-6,
    ensure_symmetry: bool = False
) -> torch.Tensor:
    """Project a matrix onto the positive definite (PD) cone via eigen-decomposition.

    The procedure ensures the output is symmetric and positive semidefinite by:
    
    1. Optionally symmetrizing the input matrix.
    2. Performing eigen-decomposition.
    3. Clipping eigenvalues below ``eps`` to enforce non-negativity.
    4. Reconstructing the matrix from clipped eigenvalues and eigenvectors.
    5. Symmetrizing the result again to avoid numerical drift.

    Args:
        mat (torch.Tensor): Input square matrix of shape ``(k, k)``.
        eps (float, optional): Minimum eigenvalue threshold to enforce positive definiteness.
            Defaults to ``1e-6``.
        ensure_symmetry (bool, optional): If True, symmetrize the input before decomposition.
            Defaults to False.

    Returns:
        torch.Tensor: Symmetric positive semidefinite matrix of shape ``(k, k)``.

    Example:
        >>> M = torch.tensor([[1.0, 2.0], [2.0, -3.0]])
        >>> M_psd = eigen_decomp_proj_to_pd(M)
        >>> torch.linalg.eigvalsh(M_psd)
        tensor([1.0133e-06, 1.8284e+00])
    """
    # Ensure symmetry
    if ensure_symmetry:
        mat = (mat + mat.T) / 2
    
    # Eigen-decomposition
    eigvals, eigvecs = torch.linalg.eigh(mat)
    
    # Clip eigenvalues to non-negative
    eigvals_clipped = torch.clamp(eigvals, min=eps)
    
    # Reconstruct
    mat_psd = eigvecs @ torch.diag(eigvals_clipped) @ eigvecs.T
    
    # Ensure symmetry again
    return (mat_psd + mat_psd.T) / 2

def generate_sigma_bad_and_good(
    k: int,
    proportion_var_dif: float,
    generator: torch.Generator,
    var_range: Tuple[float, float] = (0.0, 1.0),
    eps: float = 1e-6,
    device: torch.device = torch.device("cpu"),
    dtype: torch.dtype = torch.float64
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Generate a pair of covariance matrices: one 'good' baseline and one 'bad' perturbed version.

    The construction proceeds as follows:

    1. Generate two baseline covariance matrices using ``random_vcov_matrix``.
    2. Sample a random mask over the upper-triangular entries (including diagonal).
    3. Copy selected entries from the 'good' matrix into the 'bad' matrix, leaving
       others perturbed.
    4. Reflect the upper-triangular entries to the lower-triangular part to ensure symmetry.
    5. Project the 'bad' matrix onto the positive definite cone using
       :func:`eigen_decomp_proj_to_pd`.

    Args:
        k (int): Dimension of the covariance matrices.
        proportion_var_dif (float): Probability of keeping an entry different between
            the 'bad' and 'good' matrices.
        generator (torch.Generator): Random number generator for reproducibility.
        var_range (Tuple[float, float], optional): Range for diagonal variances.
            Defaults to (0.0, 1.0).
        eps (float, optional): Small diagonal perturbation to ensure positive definiteness.
            Defaults to ``1e-6``.
        device (torch.device, optional): Device for tensor allocation. Defaults to CPU.
        dtype (torch.dtype, optional): Data type of the returned tensors. Defaults to ``torch.float64``.

    Returns:
        Tuple[torch.Tensor, torch.Tensor]:
            - ``sigma_bad``: Perturbed covariance matrix of shape ``(k, k)``, projected to PSD.
            - ``sigma_good``: Baseline covariance matrix of shape ``(k, k)``.

    Example:
        >>> g = torch.Generator().manual_seed(123)
        >>> sigma_bad, sigma_good = generate_sigma_bad_and_good(3, 0.5, generator=g)
        >>> sigma_bad.shape, sigma_good.shape
        (torch.Size([3, 3]), torch.Size([3, 3]))
    """
    # Step 1: Generate baseline matrices
    sigma_bad = random_vcov_matrix(k, generator=generator, var_range=var_range, device=device, dtype=dtype, eps=eps)
    sigma_good = random_vcov_matrix(k, generator=generator, var_range=var_range, device=device, dtype=dtype, eps=eps)

    # Step 2: Random mask for off-diagonal entries
    count_possible_changes = (k**2 + k) // 2 #Count diagonal entries + upper triangle
    index_change_vars = ~torch.bernoulli(torch.full((count_possible_changes,), proportion_var_dif, device=device), generator=generator).bool()

    triu_indices = torch.triu_indices(k, k, offset=0)
    indices_to_copy_sigma_bad = (triu_indices[0][index_change_vars], triu_indices[1][index_change_vars])

    sigma_good[indices_to_copy_sigma_bad] = sigma_bad[indices_to_copy_sigma_bad]
    i, j = torch.tril_indices(k, k, offset=-1)
    sigma_good[i, j] = sigma_good[j, i] # ensure symmetry
    
    sigma_good = eigen_decomp_proj_to_pd(sigma_good, eps=eps)

    return sigma_bad, sigma_good

def mvn_random_sample(
        mean : torch.Tensor, 
        cov_chol_decomp : Optional[torch.Tensor],
        n : int, 
        rng : Optional[torch.Generator] = None, 
        args_checks : bool = True
    ):
    """Generate n-vectors sampled of a multivariate normal (MVN) distribution with parameters
    mean and cov. Based on the implementation of (r)sample from 
    torch.distributions.MultivariateNormal according to torch version 2.9.1. It uses
    cholesky-decomposition method.

    Args:
        mean (torch.Tensor): Location parameter of a MVN. Shape ``(k,)`` or ``(b, k)`` or ``(1,5)``.
        cov_chol_decomp (torch.Tensor): Variance-Covariance matrix of MVN after cholesky decomposition. 
            Shape ``(k,k)``` or ``(b, k, k)`` or ``(1, k, k)``, ``cov.dim()==mean.dim()+1`` should hold.
        n (int): Count of vectors to be sampled (per batch).
        rng (Optional[torch.Generator]): If passed, sampling is done using this
            generator.
        args_checks (bool): If true, it will be checked whether the shapes of mean and
            cov are as expected, whether symmetry (w. r. t. to the last wo dims for each beach)
            is given within the range of ``symmetry_rtol_atol`` for ``cov`` and type checks
            are done for ``n`` and ``rng``.`
        symmetry_rtol_atol (Tuple[float,float]): Corresponds to the (rtol, a_tol) parameters
            of ``torch.allclose``, passed as ``*args``, so ordering is important. Ignored if
            ``not args_checks``.
    Returns:
        torch.Tensor:
            A tensor of shape ``(n, k)`` or ``(n, b, k)`` containing the ``n`` sampled vectors (for each batch).

    Example:
        >>> count_covariates = 5
        >>> device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        >>> torch.set_default_dtype(torch.float32)
        >>> rng = torch.Generator(device)
        >>> mu = torch.zeros(count_covariates)
        >>> sigma_bad, _ = generate_sigma_bad_and_good(k = count_covariates, proportion_var_dif=1.0, generator = rng, device=device, dtype= torch.get_default_dtype())
        >>> sample = mvn_random_sample(mean=mu, cov=sigma_bad, n=100, rng=rng)
        >>> sample.shape
        torch.Size([100, 5])
    """
    if args_checks:
        #shape checks
        assert (mean.dim() in [1, 2, 3]) and (cov_chol_decomp.dim()==mean.dim()+1), "mean must be a single vector (rank 1 tensor) and cov a matrix (rank 2 tensor)"
        assert (mean.size(-1) == cov_chol_decomp.size(-1)) and (cov_chol_decomp.size(-1) == cov_chol_decomp.size(-2)), "mean must have shape [k] and cov shape [k, k]"
        #ensure n is an int
        n = int(n)
        assert isinstance(rng, torch.Generator) or rng is None, "rng needs to be None or a rng"

    shape = torch.Size([n]) + mean.shape
    
    eps = torch.empty(shape, dtype=mean.dtype, device = mean.device).normal_(generator=rng)
    cov_chol_decomp = torch.linalg.cholesky(cov) # occurs over the last dimension

    deviations = torch.matmul(cov_chol_decomp, eps.unsqueeze(-1)).squeeze(-1) # apply decomp to each sampled vector


    return mean + deviations


# Extending CreditDataSample

Notes and experiments on how to keep tabs of the feature ids

In [5]:
import torch
from typing import Optional
import os
import sys
base_path = os.path.abspath("./..")
if base_path not in sys.path:
    sys.path.append(base_path)

from berebasl.simulation.credit_data_simulation import CreditDataSample


B, N_r, N_a, C = 32, 500, 200, 5

data = sample = CreditDataSample(
    features_rejects=torch.randn(B, N_r, C),
    features_accepts=torch.randn(B, N_a, C),
    default_flag_accepts=torch.randint(0,2, (B, N_a), dtype=torch.get_default_dtype())
)

In [17]:
nan_val = -1
inferred_labels = torch.distributions.Categorical(torch.tensor([0.8,0.1,0.1])).sample(data.features_unlabeled.shape[:-1]) - 1
mask_inferred_rej_lbls = inferred_labels != -1

self = sample
inplace : bool = True
safety_checks : bool = True

In [ ]:
from typing import Dict, Tuple, Union
def _mask2d_to_int_idxs(mask : torch.Tensor, correction_last_idx : Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
    mask_int = mask.to(torch.int32)
    idx_last_axis = torch.where(# [B, N]
            mask,
            mask_int.cumsum(-1) - 1,
            -1
    )
    if correction_last_idx is not None:
        idx_last_axis = idx_last_axis + correction_last_idx
    batch_idx = torch.arange(mask.size(0), device=mask.device).unsqueeze(-1).expand(mask.shape)
    
    return batch_idx[mask], idx_last_axis[mask]

if True:
    if True:
        if safety_checks:
            if mask_inferred_rej_lbls.dtype != torch.bool:
                raise ValueError("mask_infered_rej_lbls should be of type bool")
            if not (mask_inferred_rej_lbls.shape == inferred_labels.shape == self.features_unlabeled.shape[:-1]):
                raise ValueError("mask_inferred_rej_lbls and inferred_labels should have the same shape as self.features_unlabeled.shape[:-1]")
            
            inferred_labels_with_vals_as_saved_labels = torch.isin(inferred_labels[mask_inferred_rej_lbls].unique(), self.labels).all()
            if not inferred_labels_with_vals_as_saved_labels:
                raise ValueError("Tensor inferred_labels[mask_inferred_rej_lbls] should contain only values like in self.labels")
            
        
        *batch_shape, N_labels = self.labels.shape
        B = torch.tensor(batch_shape).prod()

        current_labels = self.labels.reshape(B, N_labels)
        mask_nans_labels = current_labels.isnan()

        N_unlabeled = mask_inferred_rej_lbls.size(-1)
        mask_inf = mask_inferred_rej_lbls.reshape(B, N_unlabeled)
        inf_lbls = inferred_labels.reshape(B, N_unlabeled)

        slots_available = mask_nans_labels.sum(dim=-1)
        slots_needed = mask_inf.sum(dim=-1)

        needed_padding = torch.maximum(slots_needed - slots_available, torch.tensor(0))
        max_needed_padding = needed_padding.max()

        # specifics
        pad_mode_val = {"mode" : 'constant', 'value':torch.nan}
        mask_valid_lbls = ~mask_nans_labels
        batch_idx_valid_lbls, N_idx_valid_lbls = _mask2d_to_int_idxs(mask_valid_lbls)
        batch_idx_inf_lbls, N_idx_inf_lbls = _mask2d_to_int_idxs(mask_inf, N_labels - slots_available.unsqueeze(-1))

        def _append_obs(append_to : torch.Tensor, to_append : torch.Tensor, pad_spec : tuple[int]):
            appended = torch.nn.functional.pad(append_to, pad=pad_spec, **pad_mode_val)
            appended[batch_idx_valid_lbls, N_idx_valid_lbls] = append_to[mask_valid_lbls].to(appended.dtype)
            appended[batch_idx_inf_lbls, N_idx_inf_lbls] = to_append[mask_inf].to(appended.dtype)

        new_labels = _append_obs(append_to=current_labels, to_append=inf_lbls, pad_spec=(0, max_needed_padding))
        new_inferred_ids = _append_obs(
            append_to=self._inferred_ids.reshape(B, N_labels), 
            to_append=self._unlabeled_ids.reshape(B, N_unlabeled),
            pad_spec=(0, max_needed_padding)
        )
        new_features_labeled = _append_obs(
            append_to=self.features_labeled.reshape(B, N_labels, -1), 
            to_append=self.features_unlabeled.reshape(B, N_unlabeled, -1),
            pad_spec=(0, 0, 0, max_needed_padding)
        )

        new_N_unlabeled = N_unlabeled - slots_needed.max()
        mask_non_inferred = ~mask_inferred_rej_lbls
        batch_idx_resized_unlbld, N_idx_resized_unlbld = _mask2d_to_int_idxs(mask_non_inferred)
        def _resize_obs(to_resize : torch.Tensor, pad_spec):
            resized = to_resize.new_full(torch.Size([B]) + to_resize.shape[2:], fill_value=torch.nan)
            resized[batch_idx_resized_unlbld, N_idx_resized_unlbld] = to_resize[mask_non_inferred]
            return resized

        features_unlabeled = _resize_obs(self.features_unlabeled)
        unlabeled_ids = _resize_obs(self._unlabeled_ids)

        new_labels, new_inferred_ids, new_features_labeled, features_unlabeled, unlabeled_ids = [
            t.reshape(torch.Size(batch_shape) + t.shape[1:]) for t in 
            new_labels, new_inferred_ids, new_features_labeled, features_unlabeled, unlabeled_ids
        ]
            
        if inplace:
            self.features_labeled = new_features_labeled
            self.labels = new_labels
            self._inferred_ids = new_inferred_ids

            self.features_unlabeled = features_unlabeled
            self._unlabeled_ids = unlabeled_ids
            #return
        new_instance = CreditDataSample(
            features_rejects=features_unlabeled,
            features_accepts=new_features_labeled,
            default_flag_accepts=new_labels,
            ids_rejects=unlabeled_ids
        )
        new_instance._inferred_ids = new_inferred_ids
        #return new_instance

                                                                    
nan_val = -1
inferred_labels = torch.distributions.Categorical(torch.tensor([0.8,0.1,0.1])).sample(data.features_unlabeled.shape[:-1]) - 1
mask_inferred_rej_lbls = inferred_labels != -1


## Make sure things work as expected

In [2]:
import torch
from typing import Optional
import os
import sys
base_path = os.path.abspath("./..")
if base_path not in sys.path:
    sys.path.append(base_path)

from berebasl.simulation.credit_data_simulation import CreditDataSample


B, N_r, N_a, C = 32, 500, 200, 5

data = sample = CreditDataSample(
    features_rejects=torch.randn(B, N_r, C),
    features_accepts=torch.randn(B, N_a, C),
    default_flag_accepts=torch.randint(0,2, (B, N_a), dtype=torch.int8)
)

In [ ]:
idx_labeling = 0

while idx_labeling < 25 and (~data._ids_rejects_nan_checker(data._unlabeled_ids)).sum() > 0:
    probs_inferred_lables = torch.where(
        data._ids_rejects_nan_checker(data._unlabeled_ids).unsqueeze(-1).expand(*data._unlabeled_ids.shape, 3),
        torch.tensor([1.0,0.0,0.0]).expand(*data._unlabeled_ids.shape, 3),
        torch.tensor([0.8,0.1,0.1]).expand(*data._unlabeled_ids.shape, 3)
    )
    inferred_labels = torch.distributions.Categorical(probs_inferred_lables).sample() - 1
    mask_inferred_rej_lbls = inferred_labels != -1

    # Not doing inplace as with safety checks all shape and nans-checks are done.
    data = data.label_rejects(inferred_labels, mask_inferred_rej_lbls, inplace=False, safety_checks=True) 

    if not mask_inferred_rej_lbls.any():
        break

    idx_labeling += 1